# 02 · Feature engineering
Three domain features, each tested by 5-fold CV ablation on the training split.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 20)
from ml import config
report = lambda name: json.loads((config.REPORTS_DIR / name).read_text())

In [2]:
from ml.features.engineering import ENGINEERED_FEATURES, FeatureBuilder
pd.DataFrame(ENGINEERED_FEATURES).T

,formula,reason
temp_diff_k,process_temp_k - air_temp_k,Small process-air temperature gap indicates po...
power_w,torque_nm * rotational_speed_rpm * 2*pi/60,Mechanical power; failures are associated with...
strain_nm_min,tool_wear_min * torque_nm,"Accumulated load on a worn tool, a proxy for o..."


In [3]:
from ml.preprocessing.data import load_split, feature_target
X, y = feature_target(load_split('train'))
FeatureBuilder().transform(X).head()

,type_ordinal,air_temp_k,process_temp_k,rotational_speed_rpm,torque_nm,tool_wear_min,temp_diff_k,power_w,strain_nm_min
0,1.0,298.1,308.6,1551.0,42.8,0.0,10.5,6951.590560,0.0
1,0.0,298.1,308.5,1498.0,49.4,5.0,10.4,7749.387543,247.0
2,0.0,298.2,308.6,1433.0,39.5,7.0,10.4,5927.504659,276.5
3,0.0,298.1,308.6,1558.0,42.4,14.0,10.5,6917.703247,593.6
4,1.0,298.3,308.7,1667.0,28.6,18.0,10.4,4992.639989,514.8


## Leakage check
Identifiers and the five failure-mode flags never reach the model.

In [4]:
forbidden = {config.TARGET, *config.FAILURE_MODES, *config.IDENTIFIER_COLUMNS}
assert forbidden.isdisjoint(FeatureBuilder().feature_names)
FeatureBuilder().feature_names

['type_ordinal', 'air_temp_k', 'process_temp_k', 'rotational_speed_rpm', 'torque_nm', 'tool_wear_min', 'temp_diff_k', 'power_w', 'strain_nm_min']

## Ablation (CV PR-AUC, default hyper-parameters)

In [5]:
abl = report('feature_ablation.json')
pd.DataFrame({m: {s: v['cv_pr_auc_mean'] for s, v in r.items()} for m, r in abl['results'].items()})

,logistic_regression,random_forest,hist_gradient_boosting,xgboost
raw_only,0.4386,0.7734,0.7461,0.7556
raw+temp_diff_k,0.4389,0.8290,0.8073,0.7938
raw+power_w,0.4654,0.8245,0.7844,0.7718
raw+strain_nm_min,0.4372,0.8070,0.7843,0.8048
raw+all_engineered,0.4667,0.9095,0.8593,0.8476


## Marginal value for the selected model (train CV and validation)

In [6]:
pd.DataFrame(abl['selected_model_marginal']).T[['cv_pr_auc_mean','cv_pr_auc_std','validation_pr_auc']]

,cv_pr_auc_mean,cv_pr_auc_std,validation_pr_auc
selected_set,0.9109,0.0145,0.8989
raw_only,0.7821,0.0399,0.7933
without_temp_diff_k,0.8742,0.022,0.8894
without_power_w,0.8765,0.0163,0.8461
without_strain_nm_min,0.8884,0.0227,0.8828
